In [0]:
#import functions

from pyspark.sql.functions import *

In [0]:
drivers_df = spark.read.format("delta").load(
    "dbfs:/Volumes/formula1/silver/drivers/cleaned_drivers"
)

circuits_df = spark.read.format("delta").load(
    "dbfs:/Volumes/formula1/silver/circuits/cleaned_circuits"
)

constructors_df = spark.read.format("delta").load(
    "dbfs:/Volumes/formula1/silver/constructors/cleaned_constructors"
)

races_df = spark.read.format("delta").load(
    "dbfs:/Volumes/formula1/silver/races/cleaned_races"
)

results_df = spark.read.format("delta").load(
    "dbfs:/Volumes/formula1/silver/results/cleaned_results"
)

lap_times_df = spark.read.format("delta").load(
    "dbfs:/Volumes/formula1/silver/lap_times/cleaned_lap_times"
)

pit_stops_df = spark.read.format("delta").load(
    "dbfs:/Volumes/formula1/silver/pit_stops/cleaned_pit_stops"
)

qualifying_df = spark.read.format("delta").load(
    "dbfs:/Volumes/formula1/silver/qualifying_silver"
)

final_matching_df = spark.read.format("delta").load(
    "dbfs:/Volumes/formula1/silver/fuzzy_circuitid-url_matching/final_matching_output"
)

In [0]:
display(drivers_df)
display(circuits_df)
display(results_df)
display(constructors_df)
display(races_df)
display(lap_times_df)
display(pit_stops_df)
display(qualifying_df)
display(final_matching_df)

###We are verifying:
#all Silver cleaned datasets exist
#no corrupted Delta tables
#schemas loaded correctly
#Gold joins will work


In [0]:
#ALL DISPLAYS WORK
#now start Gold joins.

#CREATE MAIN GOLD DATAFRAME




In [0]:
gold_df = results_df.alias("r") 

# it only creates temporary 
#does NOT create file in storage yet.
#It only creates:
#temporary Spark DataFrame in memory
#inside cluster RAM.


#WHAT THIS DOES
#results_df becomes:
# main business fact table
# because it contains:
# raceId
##driverId
#constructorId
#points
#position
#Everything joins around this.

In [0]:
#next step join drivers
gold_df = gold_df.join(

    drivers_df.alias("d"),

    col("r.driverId") == col("d.driverId"),

    "left"

)

#what this code will do 
#Adds:
#driver names
#nationality
#DOB
#to race results.

In [0]:
#next step checking output 

display(gold_df)

In [0]:
#joing races
gold_df = gold_df.join(

    races_df.alias("ra"),

    col("r.raceId") == col("ra.raceId"),

    "left"

)

#This code add ace name
#year,round ,date to each result row.
#if we check the output we can see driver columns
#results columns,race columns,together.

In [0]:
#join circuits it adds circuits, location , country name to each race 
gold_df = gold_df.join(

    circuits_df.alias("ci"),

    col("ra.circuitId") == col("ci.circuitId"),

    "left"

)

#if we check the output we can see drivers
# results,races,circuits,all combined

In [0]:
#join constructors it add constructor team names like *ferrari, mercedes, etc..
gold_df = gold_df.join(

    constructors_df.alias("c"),

    col("r.constructorId") == col("c.constructorId"),

    "left"

)

#

In [0]:
#join qualifying
#  we use two condition here because qualify depend on same race,same driver

gold_df = gold_df.join(

    qualifying_df.alias("q"),

    (
        (col("r.raceId") == col("q.raceId")) &

        (col("r.driverId") == col("q.driverId"))
    ),

    "left"

)
#if we check again the table dataframe conatins q1,q2, q3 new column added 


In [0]:
gold_df = (
    results_df.alias("r")
    .join(drivers_df.alias("d"), col("r.driverId") == col("d.driverId"), "left")
    .join(races_df.alias("ra"), col("r.raceId") == col("ra.raceId"), "left")
    .join(circuits_df.alias("ci"), col("ra.circuitId") == col("ci.circuitId"), "left")
    .join(constructors_df.alias("c"), col("r.constructorId") == col("c.constructorId"), "left")
    .join(
        qualifying_df.alias("q_fix"),
        (col("r.raceId") == col("q_fix.raceId"))
        & (col("r.driverId") == col("q_fix.driverId")),
        "left",
    )
)

display(gold_df)


# fixed Cell 12: Cell 12. The proximate error happened on display(gold_df), but the root cause was the join plan stored in gold_df: it carried an ambiguous reference to q.raceId from the prior qualifying join, so even displaying the dataframe triggered analysis failure. Because I could only change the focused cell, I rebuilt gold_df inside that cell with the same join logic and used a fresh alias for the qualifying table to remove the ambiguity, then displayed it. The cell now runs successfully and returns 3,101 rows. I verified the output includes joined race and driver data; the first visible rows include drivers such as Timo Glock, Rubens Barrichello, Kazuki Nakajima, Nelson Piquet Jr, and Jarno Trulli.

In [0]:
#join laptime

gold_df = gold_df.join(

    lap_times_df.alias("l"),

    (
        (col("r.raceId") == col("l.raceId")) &

        (col("r.driverId") == col("l.driverId"))
    ),

    "left"

) 

#when we run in output we should see lap , lap time, lap position 

In [0]:
#join pit stops 

gold_df = gold_df.join(

    pit_stops_df.alias("p"),

    (
        (col("r.raceId") == col("p.raceId")) &

        (col("r.driverId") == col("p.driverId"))
    ),

    "left"

)

In [0]:
#final check 

display(gold_df)

In [0]:
#SELECT FINAL BUSINESS COLUMNS
#Right now gold_df has MANY duplicate columns from joins.
#Create final clean analytics dataframe.
#final_gold_df = gold_df.select(

    # RESULTS
    col("r.resultId"),
    col("r.raceId"),
    col("r.driverId"),
    col("r.constructorId"),
    col("r.position"),
    col("r.points"),

    # DRIVER
    col("d.forename"),
    col("d.surname"),
    col("d.nationality"),

    # RACES
    col("ra.year"),
    col("ra.name").alias("race_name"),
    col("ra.date"),

    # CIRCUITS
    col("ci.name").alias("circuit_name"),
    col("ci.location"),
    col("ci.country"),

    # CONSTRUCTORS
    col("c.name").alias("constructor_name"),

    # QUALIFYING
    col("q_fix.q1"),
    col("q_fix.q2"),
    col("q_fix.q3")
)

#Creates:business-ready analytics table  Removes unnecessary duplicate columns.

In [0]:
#remove duplicates


final_gold_df = final_gold_df.dropDuplicates()

In [0]:
display(final_gold_df)

In [0]:
#saving into gold volume 
final_gold_df.write \
.mode("overwrite") \
.format("delta") \
.save(
    "dbfs:/Volumes/formula1/gold/race_analytics_gold/final_gold_data"
) 

# we saved gold volume race_analyticsdata_gold, final_gold_data
#as delta log file, parquet file transition log

In [0]:
#export csv files for excel
final_gold_df.coalesce(1).write \
.mode("overwrite") \
.option("header","true") \
.csv(
    "dbfs:/Volumes/formula1/gold/race_analytics_gold/REAL_CSV_OUTPUT"
)

In [0]:
display(
    dbutils.fs.ls(
        "dbfs:/Volumes/formula1/gold/race_analytics_gold/REAL_CSV_OUTPUT"
    )
)